# Weakly Supervised 3D Disease Localization — Google Colab

BB annotasyonları (2D, kesit bazlı) + TotalSegmentator organ maskeleri →  
**Hastalığın 3D lokalizasyonu** (weakly supervised, pseudo-label değil)

## Yöntem
1. TotalSegmentator **sadece inference** → 6 anatomik organ maskesi
2. BB annotasyonlu kesitlerde: `BB_bölgesi ∩ organ_maskesi`
3. Boundary Slice z-aralığındaki diğer kesitlerde: `organ_maskesi`
4. Dışında: background (0)
5. Çıktı: çok-sınıflı 3D NIfTI  (0=bg, 1..6=hastalık)

## Google Drive Klasör Yapısı
```
MyDrive/abdomen/
    Bilgi.xlsx
    Egitim Verisi/
    Yarisma Veri Seti/
    abdomen_project/
        src/   scripts/
        outputs/splits/  <- manifest.csv burada
```

## [2] Paket Kurulumu

In [ ]:
%%capture

In [ ]:
!pip install -q TotalSegmentator SimpleITK pydicom pandas openpyxl tqdm scikit-learn

In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.5 MB 3.3 MB/s eta 0:00:02
   --------- ------------------------------ 1.3/5.5 MB 4.5 MB/s eta 0:00:01
   ----------- ---------------------------- 1.6/5.5 MB 3.1 MB/s eta 0:00:02
   -------------------------- ------------- 3.7/5.5 MB 4.9 MB/s eta 0:00:01
   ---------------------------------- ----- 4.7/5.5 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 5.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 6.0 MB/s eta 0:07:51
   ---------------------------------------- 0.0/2.8 GB 6.0 MB/s eta 0:07:51
   ---------------------------------------- 0.0/

In [1]:
import sys
!{sys.executable} -m pip install -q TotalSegmentator 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.0.7 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 9.1.0 which is incompatible.


In [ ]:
# import sys
# !{sys.executable} -m pip install SimpleITK --only-binary=:all:

In [6]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

print(f"Ortam: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Yerel'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı.")

Ortam: Yerel


In [5]:

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f'GPU: {name} ({mem} GB)')
else:
    print('GPU yok — Runtime > T4 GPU secin.')

GPU yok — Runtime > T4 GPU secin.


## [1] Kullanıcı Ayarları — yalnızca bu hücreyi düzenleyin

In [7]:
import os, sys, shutil, subprocess, json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List

# ── Ortam bazlı yol konfigürasyonu ────────────────────────────────────────
if IS_KAGGLE:
    KAGGLE_INPUT = Path('/kaggle/input/datasets/ramazan2020/abdomen-acikveri')
    DATA_ROOT    = KAGGLE_INPUT
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DATA_ROOT / "outputs/splits"
    MASK_DIR    = DATA_ROOT / "outputs/weak_disease_masks"
    CODE         = DATA_ROOT

elif IS_COLAB:
    DRIVE_BASE   = Path('/content/drive/MyDrive/abdomen')
    DATA_ROOT    = DRIVE_BASE
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DRIVE_BASE / "outputs/splits"
    MASK_DIR    = DRIVE_BASE / "outputs/weak_disease_masks"
    CODE         = DRIVE_BASE

else:
    from dotenv import load_dotenv
    load_dotenv()
    DATA_ROOT    = Path(os.environ.get("TR_ABDOMEN_BASE", "."))
    PROJE        = Path(os.environ.get("TR_ABDOMEN_PROJE", "."))
    CODE         = Path(os.environ.get("TR_ABDOMEN_CODE",  "."))
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = PROJE / "outputs/splits"
    MASK_DIR    = PROJE / "outputs/weak_disease_masks"


if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))



SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0 → label 1
    "kidney_ureter_stone",        # 1 → label 2
    "acute_pancreatitis",         # 2 → label 3
    "aortic_aneurysm_dissection", # 3 → label 4
    "acute_appendicitis",         # 4 → label 5
    "acute_diverticulitis",       # 5 → label 6
]






In [8]:
TOTALSEG_FAST = True   # T4 16GB icin onerilir
LIMIT         = None   # Debug: kac vaka (None=hepsi)

for p in [DATA_ROOT, PROJE, SPLIT_DIR, EGITIM_DIR]:
    print(f'  [{"OK" if p.exists() else "EKSIK"}] {p}')

  [OK] D:\makale-pdf\Proje\abdomen
  [OK] D:\makale-pdf\Proje
  [OK] D:\makale-pdf\Proje\outputs\splits
  [OK] D:\makale-pdf\Proje\abdomen\Egitim Verisi


In [9]:
import SimpleITK as sitk, pydicom
print(f'SimpleITK : {sitk.Version()}')
print(f'pydicom   : {pydicom.__version__}')
!TotalSegmentator --version

SimpleITK : SimpleITK Version: 2.5.5 (ITK 5.4)
Compiled: May 12 2026 17:33:28

pydicom   : 3.0.2
2.13.0


## [3] Proje Kodunu Colab Diskine Kopyala

In [ ]:
# import shutil, sys
# COLAB_PROJECT = Path('/content/abdomen_project')
# if COLAB_PROJECT.exists():
#     shutil.rmtree(COLAB_PROJECT)
# shutil.copytree(str(PROJE), str(COLAB_PROJECT))
# if str(COLAB_PROJECT) not in sys.path:
#     sys.path.insert(0, str(COLAB_PROJECT))
# print(f'Proje kopyalandi: {COLAB_PROJECT}')

## [5] Manifest DICOM Yollarını Yeniden Eşle

In [ ]:
# import shutil
# import pandas as pd

# manifest_src = SPLIT_DIR / 'manifest.csv'
# assert manifest_src.exists(), f'manifest.csv bulunamadi: {manifest_src}'

# df     = pd.read_csv(manifest_src)
# sample = str(df['dicom_path'].iloc[0])
# print(f'Orjinal yol: {sample}')

# COLAB_SPLITS = Path('/content/splits_colab')
# COLAB_SPLITS.mkdir(exist_ok=True)

# if not (sample.startswith('/content/drive') or sample.startswith(str(DATA_ROOT))):
#     drive_parent = str(DATA_ROOT.parent)
#     def _remap(p):
#         p = str(p)
#         return (drive_parent + '/' + p[p.index('abdomen'):]) if 'abdomen' in p else p
#     df['dicom_path'] = df['dicom_path'].apply(_remap)
#     print(f'Yeniden eslendi: {df["dicom_path"].iloc[0]}')

# df.to_csv(COLAB_SPLITS / 'manifest.csv', index=False)
# for f in SPLIT_DIR.glob('*.csv'):
#     if f.name != 'manifest.csv':
#         shutil.copy(f, COLAB_SPLITS / f.name)

# os.environ['ABDOMEN_SPLIT_DIR'] = str(COLAB_SPLITS)
# bb_count = (df['bboxes'].fillna('') != '').sum()
# print(f'Toplam satir: {len(df):,}  |  BB annotasyonlu: {bb_count:,}')

## [6] 06b_weak_seg.py — Weakly Supervised 3D Hastalık Maskesi

```
TotalSegmentator (inference)
    -> 6 organ maskesi (3D)
    x  BB annotasyonlari (2D)
    =  3D hastalik maskesi  (0=bg, 1=kolesistit, ..., 6=divertikulit)
```

> **Sure:** ~3-8 dk/vaka (T4, fast mod). 735 egitim vakasi ~ 40-100 saat toplam.  
> Debug icin `LIMIT = 5` kullanin.
>
> **Not:** Apendiks TotalSegmentator'da ayri sinif degildir; kolon maskesinden yaklasik alinir.

In [ ]:
from src.weak_seg import generate_weak_masks

generate_weak_masks(
    limit=LIMIT,
    out_dir=MASK_DIR,
    totalseg_fast=TOTALSEG_FAST,
)

TotalSeg+Maske:  29%|██▉       | 191/652 [4:14:50<9:39:27, 75.42s/it]  

## [7] Sonuçları Google Drive'a Kaydet

In [ ]:
# import shutil
# DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
# masks = list(MASK_DIR.glob('*_disease.nii.gz'))
# for p in masks:
#     dst = DRIVE_RESULTS / p.name
#     if not dst.exists():
#         shutil.copy(p, dst)
# stats_src = MASK_DIR / 'stats.json'
# if stats_src.exists():
#     shutil.copy(stats_src, DRIVE_RESULTS / 'stats.json')
# saved = list(DRIVE_RESULTS.glob('*_disease.nii.gz'))
# print(f'Kaydedildi: {DRIVE_RESULTS}')
# print(f'Toplam maske: {len(saved)}')

## [8] İstatistikler & Görselleştirme

In [ ]:
%matplotlib inline
import json
import matplotlib.pyplot as plt

with open(MASK_DIR / 'stats.json') as f:
    stats = json.load(f)

classes = list(stats['voxel_counts'].keys())
counts  = list(stats['voxel_counts'].values())
print(f'Islenen vaka: {stats["processed_cases"]}')
for cls, n in zip(classes, counts):
    print(f'  {cls:35s}: {n:>12,}')

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
bars = ax.barh(classes, counts, color=colors, alpha=0.85)
ax.bar_label(bars, labels=[f'{n:,}' for n in counts], padding=4, fontsize=9)
ax.set_xlabel('Toplam voxel sayisi')
ax.set_title('Weakly Supervised 3D Hastalik Maskesi — Sinif Dagilimi')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## [6] CT + Maske Görselleştirme

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from src.config import SUPER_CLASSES

COLORS = [None,'#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
masks  = sorted(MASK_DIR.glob('*_disease.nii.gz'))

if not masks:
    print('Gorsellestirilecek maske yok.')
else:
    mask_path = masks[len(masks) // 2]
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path)))
    dis_slices = np.where(arr.max(axis=(1, 2)) > 0)[0]

    if len(dis_slices) == 0:
        print(f'{mask_path.stem}: hastalık maskesi bos.')
    else:
        mid_z   = dis_slices[len(dis_slices) // 2]
        sl      = arr[mid_z]
        overlay = np.zeros((*sl.shape, 4), dtype=np.float32)
        patches = []
        for cid in range(1, 7):
            region = sl == cid
            if not region.any():
                continue
            rgba = mcolors.to_rgba(COLORS[cid])
            overlay[region] = (*rgba[:3], 0.7)
            patches.append(mpatches.Patch(color=COLORS[cid],
                           label=SUPER_CLASSES[cid - 1].replace('_', ' ')))

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow((sl == 0).astype(float), cmap='gray', alpha=0.4)
        ax.imshow(overlay)
        if patches:
            ax.legend(handles=patches, loc='upper right', fontsize=8)
        ax.set_title(f'{mask_path.stem}  |  z={mid_z}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

In [13]:
from src.dicom_utils import load_series, window_hu, Window
from src.config import SUPER_CLASSES, RAW_TRAIN_DIR, RAW_TEST_DIR


## [7] Analiz: BB Coverage & Z-Range Uyumu

In [ ]:
import pandas as pd
from src.config import RAW_PATHOLOGY_TO_SUPER, SUPER_CLASSES

sheets  = pd.read_excel(DATA_ROOT / 'Bilgi.xlsx', sheet_name=None)
all_ann = pd.concat([
    sheets['TRAIININGDATA'].assign(source='train'),
    sheets['COMPETITIONDATA'].assign(source='comp'),
], ignore_index=True)

def parse_bb(raw):
    try:
        a,b = raw.split('-')
        x1,y1 = map(int,a.split(','))
        x2,y2 = map(int,b.split(','))
        return (x1,y1,x2,y2) if x2>x1 and y2>y1 else None
    except: return None

records = []
for mask_path in sorted(MASK_DIR.glob('*_disease.nii.gz')):
    cid = int(mask_path.stem.split('_')[1])
    case_dir_ = next((b/str(cid) for b in (RAW_TRAIN_DIR,RAW_TEST_DIR)
                      if (b/str(cid)).is_dir()), None)
    if not case_dir_: continue
    try:
        ser_  = load_series(case_dir_)
        arr_  = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path)))
        idx_  = {img_id:i for i,img_id in enumerate(ser_.image_ids)}
        rows_ = all_ann[(all_ann['Case Number']==cid) &
                        (all_ann['Type']=='Bounding Box')]
        for _,r in rows_.iterrows():
            bb  = parse_bb(str(r['Data']))
            sid = RAW_PATHOLOGY_TO_SUPER.get(r['Class'])
            if bb is None or sid is None: continue
            z   = idx_.get(int(r['Image Id']))
            if z is None: continue
            gt   = np.zeros(arr_[z].shape,np.uint8); gt[bb[1]:bb[3],bb[0]:bb[2]]=1
            pred = (arr_[z]==sid+1).astype(np.uint8)
            tp   = (gt&pred).sum()
            records.append({
                'case'    : cid,
                'class'   : SUPER_CLASSES[sid].replace('_',' '),
                'BB_px'   : int(gt.sum()),
                'Mask_px' : int(pred.sum()),
                'TP_px'   : int(tp),
                'BB_Cov%' : round(tp/gt.sum()*100,1) if gt.sum() else 0.0,
                'MaskPr%' : round(tp/pred.sum()*100,1) if pred.sum() else 0.0,
            })
    except Exception as e:
        print(f'  [skip] {cid}: {e}')

if records:
    df_m = pd.DataFrame(records)
    print('Sınıf bazında ortalama metrikler:')
    print(df_m.groupby('class')[['BB_Cov%','MaskPr%']].mean().round(1).to_string())
    print(f'\nGenel BB Coverage  : {df_m["BB_Cov%"].mean():.1f}%')
    print(f'Genel Mask Precision: {df_m["MaskPr%"].mean():.1f}%')
    display(df_m)
else:
    print('Metrik hesaplanamadi.')

In [ ]:
!pip install -q plotly


## [8] Gerçek 3D Görselleştirme (Plotly Interactive)

Mevcut [6] bölümündeki 2D kesit görünümünün aksine bu bölüm:
- **marching cubes** ile her hastalık sınıfının 3D yüzeyini çıkarır
- **Plotly Mesh3d** ile notebook içinde döndürülebilir, zoomlanabilir 3D render yapar
- 3 adet aksiyel CT kesiti şeffaf arka plan düzlemi olarak ekler

`pip install plotly` gereklidir (bir kez).

In [12]:

import numpy as np
import SimpleITK as sitk
import plotly.graph_objects as go
from skimage.measure import marching_cubes
from src.config import SUPER_CLASSES, RAW_TRAIN_DIR, RAW_TEST_DIR
from src.dicom_utils import load_series, window_hu, Window

# ─── Ayarlar ──────────────────────────────────────────────────────────────
VIZ_CASE_3D = None   # None=ilk mevcut maske, ya da örn: 20001
DOWNSAMPLE  = 2      # 1=tam, 2=yarı (hız/bellek dengesi)
SOFT_WIN    = Window("soft_tissue", level=40, width=400)

COLORS_HEX = [
    "#e74c3c",  # acute_cholecystitis
    "#3498db",  # kidney_ureter_stone
    "#2ecc71",  # acute_pancreatitis
    "#f39c12",  # aortic_aneurysm_dissection
    "#9b59b6",  # acute_appendicitis
    "#1abc9c",  # acute_diverticulitis
]

def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

# ─── Maske yükle ──────────────────────────────────────────────────────────
masks_3d = sorted(MASK_DIR.glob("*_disease.nii.gz"))
assert masks_3d, "Maske bulunamadi -- once [4] hucresini calistirin."

mask_path = (MASK_DIR / f"ABE_{VIZ_CASE_3D:05d}_disease.nii.gz"
             if VIZ_CASE_3D else masks_3d[0])
cid = int(mask_path.stem.split("_")[1])
print(f"Vaka: {cid}  |  {mask_path.name}")

sitk_mask = sitk.ReadImage(str(mask_path))
arr       = sitk.GetArrayFromImage(sitk_mask)   # (Z, Y, X)
spacing   = sitk_mask.GetSpacing()               # (sx, sy, sz) mm
print(f"Shape: {arr.shape}, spacing: {tuple(round(s,2) for s in spacing)} mm")

d      = DOWNSAMPLE
arr_ds = arr[::d, ::d, ::d]
sp_ds  = (spacing[0]*d, spacing[1]*d, spacing[2]*d)

# ─── CT serisi ────────────────────────────────────────────────────────────
case_dir = next((b/str(cid) for b in (RAW_TRAIN_DIR, RAW_TEST_DIR)
                 if (b/str(cid)).is_dir()), None)
series   = load_series(case_dir) if case_dir else None

# ─── Plotly figure ────────────────────────────────────────────────────────
traces = []

# 1) Her hastalik sinifi icin marching cubes -> Mesh3d
for cls_id in range(1, 7):
    region = (arr_ds == cls_id).astype(np.uint8)
    if region.sum() < 8:
        continue
    verts, faces, _, _ = marching_cubes(region, level=0.5)
    vx = verts[:, 2] * sp_ds[0]
    vy = verts[:, 1] * sp_ds[1]
    vz = verts[:, 0] * sp_ds[2]
    r, g, b = _hex_to_rgb(COLORS_HEX[cls_id - 1])
    cls_name = SUPER_CLASSES[cls_id - 1].replace("_", " ")
    traces.append(go.Mesh3d(
        x=vx, y=vy, z=vz,
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=f"rgb({r},{g},{b})",
        opacity=0.75,
        name=cls_name,
        showlegend=True,
        flatshading=False,
        lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2, roughness=0.5),
    ))
    print(f"  {cls_name}: {len(verts):,} vertex, {len(faces):,} face")

# 2) Aksiyel CT kesitlerini seffaf duzlem olarak ekle
if series is not None:
    dis_zs = np.where(arr.max(axis=(1, 2)) > 0)[0]
    n = len(dis_zs)
    slice_zs = [dis_zs[n // 4], dis_zs[n // 2], dis_zs[3 * n // 4]]
    H, W = arr.shape[1], arr.shape[2]
    X2d, Y2d = np.meshgrid(np.arange(W) * spacing[0],
                           np.arange(H) * spacing[1])
    for z_idx in slice_zs:
        ct_w = window_hu(series.hu[z_idx], SOFT_WIN)
        traces.append(go.Surface(
            x=X2d, y=Y2d,
            z=np.full_like(X2d, z_idx * spacing[2]),
            surfacecolor=ct_w,
            colorscale="gray",
            showscale=False,
            opacity=0.30,
            showlegend=False,
        ))

# ─── Layout ───────────────────────────────────────────────────────────────
fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Case {cid} — 3D Hastalık Maskesi  (döndür / zoom / çift tık = sıfırla)",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data",
        bgcolor="rgb(15,15,25)",
        xaxis=dict(backgroundcolor="rgb(15,15,25)", gridcolor="#333"),
        yaxis=dict(backgroundcolor="rgb(15,15,25)", gridcolor="#333"),
        zaxis=dict(backgroundcolor="rgb(15,15,25)", gridcolor="#333"),
    ),
    paper_bgcolor="rgb(25,25,35)",
    font_color="white",
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(0,0,0,0.55)", font_size=11),
    width=950, height=720,
)
fig.show()


AssertionError: Maske bulunamadi -- once [4] hucresini calistirin.